# Notebook 01 - Analisis Exploratorio de Datos (EDA)

**MatchMind: Prediccion de Resultados de Futbol Internacional**  
Proyecto Final - Inteligencia Artificial - EAFIT 2026-1  
Integrantes: Matias Martinez Moreno (1000117104), Samuel Herrera (1000358613)

---

## Que es y para que sirve este notebook

Antes de entrenar cualquier modelo de Machine Learning hay que **conocer los datos**. Esta es la fase mas importante (y la mas subestimada) de cualquier proyecto de IA. Aqui buscamos:

1. **Entender la estructura**: cuantas filas, que columnas, que tipos de datos.
2. **Detectar problemas**: valores nulos, outliers, duplicados, errores de captura.
3. **Visualizar patrones**: distribuciones, correlaciones, evolucion temporal.
4. **Construir la variable objetivo (target)**: la columna que queremos predecir.
5. **Documentar hallazgos**: cada cosa rara que veamos nos puede ahorrar horas de debugging despues.

Trabajamos con dos datasets de Kaggle:
- `results.csv`: 49,287 partidos internacionales de futbol (1872-2026)
- `fifa_ranking.csv`: 67,472 filas del ranking FIFA historico (1992-2024)

Al final del notebook tendremos un resumen claro de cinco hallazgos que **motivaran decisiones tecnicas concretas** en los notebooks siguientes.

## 0. Configuracion del entorno

Importamos las librerias necesarias. `pandas` para manipular tablas, `matplotlib` y `seaborn` para visualizar. Tambien anadimos la raiz del proyecto al path de Python para poder importar nuestro propio codigo de `src/`.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 200

# Nuestras funciones de carga
from src.data.load import load_results, load_fifa_ranking, make_target

print('Librerias cargadas correctamente')

## 1. Cargar el dataset de partidos

El archivo `results.csv` contiene una fila por partido internacional. Cada partido tiene: fecha, equipo local, equipo visitante, goles de cada uno, tipo de torneo, ciudad y pais, y si fue en campo neutral. 

Tip de exploracion: siempre que abras un dataset nuevo, mira primero `.shape` (cuantas filas/columnas), `.head()` (primeras filas), y `.dtypes` (tipos de cada columna). Tres comandos que te ahorran muchos malentendidos.

In [ ]:
df = load_results()
print(f'Total de partidos: {len(df):,}')
print(f'Rango temporal: {df["date"].min().date()} hasta {df["date"].max().date()}')
df.head()

### Cargar tambien el ranking FIFA historico

El ranking FIFA es una metrica oficial que clasifica a las selecciones nacionales segun su rendimiento. Una seleccion como Brasil suele estar en el top 5; equipos pequenos pueden estar en el puesto 150. Vamos a usar este ranking como feature porque captura mucha informacion sobre la calidad relativa de los equipos.

In [ ]:
ranking = load_fifa_ranking()
print(f'Filas de ranking FIFA: {len(ranking):,}')
print(f'Rango: {ranking["rank_date"].min().date()} hasta {ranking["rank_date"].max().date()}')
ranking.head()

## 2. Inspeccion basica de tipos y nulos

Antes de cualquier analisis, necesitamos saber:
- **Tipos de datos** (`df.info()`): si una columna esta como string cuando deberia ser numero, tendremos problemas.
- **Valores nulos** (`df.isnull().sum()`): muchos algoritmos no toleran NaN. Hay que decidir si imputar (rellenar) o eliminar.
- **Estadisticas descriptivas** (`df.describe()`): media, min, max, percentiles. Util para detectar outliers.

In [ ]:
df.info()

In [ ]:
# Cuantos nulos hay por columna
nulls = df.isnull().sum()
nulls

**Hallazgo:** hay 72 partidos sin score (0.15% del total). 

**Decision documentada:** los eliminamos porque son solo un 0.15% y porque sin score no podemos generar la variable objetivo. Si fueran muchos mas, intentariamos investigar la causa (errores en el dataset original).

In [ ]:
df = df.dropna(subset=['home_score', 'away_score']).reset_index(drop=True)
print(f'Partidos despues de limpiar: {len(df):,}')
df.describe()

Observa el `max` de `home_score`: 31. Eso sugiere outliers (partidos con goleadas extremas). Lo investigaremos en la seccion 8.

## 3. Construir la variable objetivo (target)

El target es lo que queremos predecir. Aqui tenemos un problema de **clasificacion multiclase con 3 clases**:
- `0` = victoria visitante (away win): `home_score < away_score`
- `1` = empate (draw): `home_score == away_score`
- `2` = victoria local (home win): `home_score > away_score`

La funcion `make_target(df)` esta definida en `src/data/load.py` y usa `np.select()` para hacer esto eficientemente.

In [ ]:
df = make_target(df)

# Distribucion de clases
counts = df['result'].value_counts(normalize=True).sort_index()
counts.index = ['away_win', 'draw', 'home_win']
counts.apply(lambda x: f'{x:.2%}')

**Lectura importante:** vemos que **48.9% de los partidos los gana el local**, 22.7% son empates y 28.4% los gana el visitante. 

Esto se llama **imbalance moderado**. Significa que un modelo trivial que prediga siempre `home_win` tendria una Accuracy de 0.489 (49% de aciertos), lo cual suena bien pero NO sirve: ese modelo nunca acertaria un empate ni una victoria visitante. 

**Decision tecnica:** vamos a usar **F1-macro** como metrica principal en lugar de Accuracy. F1-macro promedia el F1 de cada clase por igual, lo que penaliza a modelos que ignoren las clases minoritarias.

## 4. Visualizacion 1: distribucion del target

Una grafica de barras de la distribucion del target debe ser SIEMPRE la primera figura del EDA en problemas de clasificacion. Confirma visualmente lo que vimos numericamente.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
labels = ['Visitante gana', 'Empate', 'Local gana']
counts_abs = df['result'].value_counts().sort_index()
ax.bar(labels, counts_abs.values, color=['#9b1b1b', '#b45309', '#006b3c'])
ax.set_title('Distribucion del resultado del partido (1872-2026)')
ax.set_ylabel('Numero de partidos')
for i, v in enumerate(counts_abs.values):
    ax.text(i, v + 200, f'{v:,}\n({v/len(df):.1%})', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('../docs/figures/eda_distribucion.png', dpi=200, bbox_inches='tight')
plt.show()

## 5. Visualizacion 2: goles totales por tipo de torneo

Hay 193 tipos de torneo distintos. Pero la mayoria de los partidos pertenecen a unos pocos. Usamos un boxplot para ver si la distribucion de goles cambia entre torneos. Si cambia mucho, el tipo de torneo puede ser una feature util.

**Que es un boxplot:** cada caja muestra el rango intercuartil (Q1 a Q3), la linea de adentro es la mediana, los bigotes muestran el resto de datos y los puntos sueltos son outliers. Es la mejor forma de comparar distribuciones rapidamente.

In [ ]:
df['total_goals'] = df['home_score'] + df['away_score']
top_tournaments = df['tournament'].value_counts().head(8).index
subset = df[df['tournament'].isin(top_tournaments)]
fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=subset, x='tournament', y='total_goals', ax=ax, order=top_tournaments)
ax.set_title('Distribucion de goles totales por tipo de torneo (top 8)')
plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../docs/figures/eda_goles_torneo.png', dpi=200, bbox_inches='tight')
plt.show()

**Lectura:** los amistosos (Friendly) tienden a tener mas goles que las finales de torneos oficiales. Esto tiene sentido: en finales los equipos juegan mas defensivo. 

**Decision:** incluiremos el tipo de torneo como feature en el modelo (one-hot encoded si es categorica).

## 6. Visualizacion 3: evolucion temporal

Muchas veces los datasets tienen sesgos temporales: pocos datos al principio, muchos al final. Esto importa porque:
1. **Para el split:** queremos que train y test tengan distribuciones parecidas.
2. **Para el modelo:** las reglas del juego cambian con el tiempo (ej: cambio de regla del fuera de juego).

In [ ]:
df['decade'] = (df['date'].dt.year // 10) * 10
fig, ax = plt.subplots(figsize=(10, 4))
df.groupby('decade').size().plot(kind='bar', ax=ax, color='#003d79')
ax.set_title('Partidos internacionales por decada')
ax.set_xlabel('Decada')
ax.set_ylabel('Numero de partidos')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../docs/figures/eda_temporal.png', dpi=200, bbox_inches='tight')
plt.show()

**Hallazgo critico:** solo hay 13 partidos en la decada de 1870 y 5,948 en la decada de 2020. Crecimiento exponencial.

**Decision:** vamos a filtrar a partidos posteriores a 1993 (cuando empieza el ranking FIFA confiable). Asi evitamos que el modelo aprenda de partidos antiguos que tienen reglas, equipos y dinamicas muy distintas. Esto se hara en el notebook 02 (preprocessing).

## 7. Visualizacion 4: ventaja de localia

Aqui exploramos una hipotesis especifica: **jugar de local da ventaja**. La columna `neutral` nos dice si el partido fue en cancha del local (False) o en cancha neutral (True). Si la ventaja existe, deberiamos ver mas victorias locales cuando `neutral == False`.

In [ ]:
result_by_neutral = df.groupby('neutral')['result'].value_counts(normalize=True).unstack()
result_by_neutral.columns = ['Visitante gana', 'Empate', 'Local gana']

fig, ax = plt.subplots(figsize=(7, 4))
result_by_neutral.plot(kind='bar', stacked=True, ax=ax, color=['#9b1b1b', '#b45309', '#006b3c'])
ax.set_title('Resultado segun si el campo es neutral')
ax.set_xticklabels(['Campo del local', 'Campo neutral'], rotation=0)
ax.set_ylabel('Proporcion')
ax.legend(loc='center left', bbox_to_anchor=(1.0, 0.5))
plt.tight_layout()
plt.savefig('../docs/figures/eda_localia.png', dpi=200, bbox_inches='tight')
plt.show()

# Numero exacto
print('\nVictorias locales segun campo:')
print(result_by_neutral['Local gana'].apply(lambda x: f'{x:.1%}'))

**Resultado:** el equipo local gana el **50.7%** de las veces en campo propio, pero solo el **43.9%** en campo neutral. 

**Diferencia: 6.8 puntos porcentuales**. Pequena pero acumulada sobre miles de partidos representa una senal estadisticamente fuerte. 

**Decision:** `neutral` sera una de las features del modelo.

## 8. Analisis de outliers en goles

Vimos que el max de `home_score` era 31. Investiguemos los partidos con muchos goles.

In [ ]:
extreme = df[df['total_goals'] > 10]
print(f'Partidos con mas de 10 goles: {len(extreme)} ({len(extreme)/len(df):.2%})')

# Top 5 goleadas
extreme.nlargest(5, 'total_goals')[['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']]

**Decision documentada:** los 287 partidos con mas de 10 goles (0.58%) **no son errores**, son partidos reales (Australia 31-0 American Samoa en clasificatorias). Los mantenemos en el dataset porque:
1. Son datos verificables historicamente.
2. Representan situaciones donde un equipo es masivamente superior, lo cual queremos que el modelo aprenda a predecir.
3. Solo son 0.58% del total, no van a dominar el entrenamiento.

## 9. Resumen de hallazgos clave

Esto es lo que llevamos al informe LaTeX y a la seccion 2 del proyecto:

| Hallazgo | Valor | Decision tecnica |
|----------|-------|------------------|
| 1. Imbalance del target | 48.9% / 22.7% / 28.4% | Usar F1-macro en vez de Accuracy |
| 2. Ventaja de localia | +6.8 pp en cancha propia | Incluir feature `neutral` |
| 3. Sesgo historico | 13 partidos en 1870s vs 5,948 en 2020s | Filtrar a post-1993 |
| 4. Outliers reales | 287 partidos con >10 goles (0.58%) | Mantener (son datos legitimos) |
| 5. Nulos de score | 72 partidos (0.15%) | Eliminar |

## Siguiente paso

Notebook **02_preprocessing.ipynb**: aplicamos feature engineering vectorizado (rachas, h2h, ranking FIFA) y hacemos el **split temporal** para evitar data leakage.